# Count-Min Sketch

## Learning Objectives

1. **Distinguish** Count-Min Sketch from AMS: point queries vs global moments
2. **Define** the $d \times w$ counter array and hash function family
3. **Derive** the additive error guarantee: $P[\hat{m}_x \leq m_x + \epsilon \|f\|_1] \geq 1 - \delta$
4. **Choose** optimal parameters $w = \lceil e/\epsilon \rceil$, $d = \lceil \ln(1/\delta) \rceil$
5. **Implement** Count-Min Sketch with configurable error and failure bounds


## Problem Statement

### Frequency Point Queries

**Query:** How many times did element $x$ appear in the stream so far?

Applications:
- **Web analytics:** count page views for each URL
- **Network monitoring:** count packets from each source IP
- **NLP:** count word frequencies in a text stream
- **Cache policy:** evict least-frequently-used items

### The Challenge

Exact counting requires $O(n)$ space (one counter per element). For large universes ($n = 2^{32}$ IPs), this is infeasible in RAM.

### Count-Min Sketch Guarantee

$$m_x \leq \hat{m}_x \leq m_x + \epsilon \|f\|_1 \quad \text{with probability} \geq 1 - \delta$$

where $\|f\|_1 = \sum_i m_i = |\text{stream}|$.

- **No undercount** (never below true frequency)
- **Additive error** bounded by $\epsilon$ fraction of total stream length
- **Space:** $O(\frac{1}{\epsilon} \log \frac{1}{\delta})$ counters


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">Count-Min Sketch (d=3 hash functions, w=8 buckets)</text>

  <!-- Row 1 (h₁) -->
  <text x="50" y="60" fill="#4e79a7" font-size="11" font-weight="bold">h₁:</text>
  <rect x="80" y="45" width="60" height="28" rx="3" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <rect x="145" y="45" width="60" height="28" rx="3" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <rect x="210" y="45" width="60" height="28" rx="3" fill="#4e79a7"/>
  <rect x="275" y="45" width="60" height="28" rx="3" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <rect x="340" y="45" width="60" height="28" rx="3" fill="#4e79a7"/>
  <rect x="405" y="45" width="60" height="28" rx="3" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <rect x="470" y="45" width="60" height="28" rx="3" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1"/>
  <rect x="535" y="45" width="60" height="28" rx="3" fill="#4e79a7"/>
  <text x="240" y="65" text-anchor="middle" fill="white" font-size="11">5</text>
  <text x="370" y="65" text-anchor="middle" fill="white" font-size="11">3</text>
  <text x="565" y="65" text-anchor="middle" fill="white" font-size="11">2</text>
  <text x="110" y="65" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="175" y="65" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="305" y="65" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="435" y="65" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="500" y="65" text-anchor="middle" fill="#ccc" font-size="11">0</text>

  <!-- Row 2 (h₂) -->
  <text x="50" y="110" fill="#f28e2b" font-size="11" font-weight="bold">h₂:</text>
  <rect x="80"  y="95" width="60" height="28" rx="3" fill="#f28e2b"/>
  <rect x="145" y="95" width="60" height="28" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1"/>
  <rect x="210" y="95" width="60" height="28" rx="3" fill="#f28e2b"/>
  <rect x="275" y="95" width="60" height="28" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1"/>
  <rect x="340" y="95" width="60" height="28" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1"/>
  <rect x="405" y="95" width="60" height="28" rx="3" fill="#f28e2b"/>
  <rect x="470" y="95" width="60" height="28" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1"/>
  <rect x="535" y="95" width="60" height="28" rx="3" fill="#fff4e0" stroke="#f28e2b" stroke-width="1"/>
  <text x="110" y="115" text-anchor="middle" fill="white" font-size="11">4</text>
  <text x="240" y="115" text-anchor="middle" fill="white" font-size="11">3</text>
  <text x="435" y="115" text-anchor="middle" fill="white" font-size="11">3</text>
  <text x="175" y="115" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="305" y="115" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="370" y="115" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="500" y="115" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="565" y="115" text-anchor="middle" fill="#ccc" font-size="11">0</text>

  <!-- Row 3 (h₃) -->
  <text x="50" y="160" fill="#59a14f" font-size="11" font-weight="bold">h₃:</text>
  <rect x="80"  y="145" width="60" height="28" rx="3" fill="#e8f8e8" stroke="#59a14f" stroke-width="1"/>
  <rect x="145" y="145" width="60" height="28" rx="3" fill="#59a14f"/>
  <rect x="210" y="145" width="60" height="28" rx="3" fill="#e8f8e8" stroke="#59a14f" stroke-width="1"/>
  <rect x="275" y="145" width="60" height="28" rx="3" fill="#59a14f"/>
  <rect x="340" y="145" width="60" height="28" rx="3" fill="#59a14f"/>
  <rect x="405" y="145" width="60" height="28" rx="3" fill="#e8f8e8" stroke="#59a14f" stroke-width="1"/>
  <rect x="470" y="145" width="60" height="28" rx="3" fill="#e8f8e8" stroke="#59a14f" stroke-width="1"/>
  <rect x="535" y="145" width="60" height="28" rx="3" fill="#e8f8e8" stroke="#59a14f" stroke-width="1"/>
  <text x="175" y="165" text-anchor="middle" fill="white" font-size="11">5</text>
  <text x="305" y="165" text-anchor="middle" fill="white" font-size="11">2</text>
  <text x="370" y="165" text-anchor="middle" fill="white" font-size="11">3</text>
  <text x="110" y="165" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="240" y="165" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="435" y="165" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="500" y="165" text-anchor="middle" fill="#ccc" font-size="11">0</text>
  <text x="565" y="165" text-anchor="middle" fill="#ccc" font-size="11">0</text>

  <!-- Query "alice" -->
  <rect x="20" y="195" width="380" height="70" rx="5" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="210" y="215" text-anchor="middle" fill="#4e79a7" font-size="12" font-weight="bold">Query count("alice")</text>
  <text x="210" y="233" text-anchor="middle" fill="#333" font-size="11">h₁("alice")=2 → C[1][2] = 5</text>
  <text x="210" y="251" text-anchor="middle" fill="#333" font-size="11">h₂("alice")=0 → C[2][0] = 4</text>
  <text x="210" y="268" text-anchor="middle" fill="#333" font-size="11">h₃("alice")=1 → C[3][1] = 5  →  min = 4 ✓</text>

  <!-- Error bound -->
  <rect x="420" y="195" width="380" height="70" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="610" y="215" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Error Bound</text>
  <text x="610" y="235" text-anchor="middle" fill="#333" font-size="11">count ≤ min_estimate ≤ count + ε||f||₁</text>
  <text x="610" y="253" text-anchor="middle" fill="#333" font-size="11">with probability 1 - δ</text>
  <text x="610" y="265" text-anchor="middle" fill="#555" font-size="10">w = ⌈e/ε⌉ buckets,  d = ⌈ln(1/δ)⌉ hash functions</text>

  <!-- Space note -->
  <rect x="20" y="280" width="780" height="68" rx="5" fill="#f5f5f5" stroke="#ccc" stroke-width="1"/>
  <text x="410" y="300" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Count-Min vs AMS Sketch</text>
  <text x="410" y="320" text-anchor="middle" fill="#555" font-size="11">Count-Min: answers point queries (count of one element). Space O(1/ε × log(1/δ)).</text>
  <text x="410" y="340" text-anchor="middle" fill="#555" font-size="11">AMS Sketch: estimates global F₂ (no per-element queries). Space O(1/ε² × log(1/δ)).</text>
</svg>
'''
display(SVG(svg))


## Derivation

### Counter Array

$d$ rows of $w$ counters: table $C[1..d][0..w-1]$, initialised to 0.

For each row $i$, a hash function $h_i : U \to [w]$ maps elements to buckets.

### Update

For stream element $x$: increment $C[i][h_i(x)]$ for all $i$.

### Query

$$\hat{m}_x = \min_{i=1}^{d} C[i][h_i(x)]$$

**No undercount:** $C[i][h_i(x)] \geq m_x$ because every occurrence of $x$ increments $C[i][h_i(x)]$.

### Error Analysis (one row)

The bucket $C[i][h_i(x)]$ may be inflated by elements $y \neq x$ with $h_i(y) = h_i(x)$ (hash collisions).

$$E[C[i][h_i(x)] - m_x] = E\left[\sum_{y \neq x} m_y \cdot \mathbf{1}[h_i(y) = h_i(x)]\right] = \frac{\|f\|_1 - m_x}{w} \leq \frac{\|f\|_1}{w}$$

By Markov's inequality: $P[C[i][h_i(x)] > m_x + \epsilon\|f\|_1] \leq \frac{1}{\epsilon w}$.

For $w = \lceil e/\epsilon \rceil$: $P[\text{row } i \text{ overestimates}] \leq 1/e$.

### Combining $d$ Rows

We take the minimum across $d$ independent rows. The query overestimates iff **all** $d$ rows overestimate:

$$P[\hat{m}_x > m_x + \epsilon\|f\|_1] \leq \left(\frac{1}{e}\right)^d = e^{-d}$$

For $d = \lceil \ln(1/\delta) \rceil$: failure probability $\leq \delta$.


## Algorithm Steps

### Build
1. Choose $d = \lceil \ln(1/\delta) \rceil$ independent hash functions $h_1, \ldots, h_d : U \to [w]$
2. Initialise $C[i][j] = 0$ for all $i, j$
3. For each stream element $x$: for $i = 1, \ldots, d$: $C[i][h_i(x)] \mathrel{+}= 1$

### Query
4. For element $x$: return $\min_{i=1}^d C[i][h_i(x)]$


In [ ]:
import numpy as np
import math


class CountMinSketch:
    """
    Count-Min Sketch for frequency estimation over a data stream.

    Inputs
    ------
    epsilon : float — additive error as fraction of stream length ||f||₁
    delta   : float — failure probability
    """

    def __init__(self, epsilon=0.01, delta=0.01):
        self.w = math.ceil(math.e / epsilon)   # buckets per row
        self.d = math.ceil(math.log(1 / delta))  # number of rows
        self.table = np.zeros((self.d, self.w), dtype=np.int64)
        self.stream_length = 0

        # Independent hash functions: h_i(x) = (a_i * x + b_i) mod p mod w
        p = 2_147_483_647
        rng = np.random.default_rng(42)
        self.a = rng.integers(1, p, size=self.d)
        self.b = rng.integers(0, p, size=self.d)
        self.p = p

        print(f"CountMin: w={self.w} buckets × d={self.d} rows = {self.w*self.d} cells")

    def _hashes(self, item):
        x = hash(item) % self.p
        return (self.a * x + self.b) % self.p % self.w

    def update(self, item, count=1):
        self.stream_length += count
        for i, bucket in enumerate(self._hashes(item)):
            self.table[i, bucket] += count

    def query(self, item):
        """
        Return upper-bound estimate of item's frequency.
        Guarantee: true_count ≤ estimate ≤ true_count + ε × ||f||₁  with prob ≥ 1-δ
        """
        return int(min(self.table[i, bucket] for i, bucket in enumerate(self._hashes(item))))


# ── Demo ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(0)

# Zipf-distributed stream
n = 100_000
vocab_size = 1_000
probs = 1.0 / np.arange(1, vocab_size + 1)
probs /= probs.sum()
stream = rng.choice(vocab_size, size=n, p=probs)

cms = CountMinSketch(epsilon=0.005, delta=0.01)
true_counts = np.zeros(vocab_size, dtype=int)

for item in stream:
    cms.update(item)
    true_counts[item] += 1

# Query top-10 most frequent items
top10 = np.argsort(true_counts)[-10:][::-1]
print(f"{'item':>6}  {'true':>8}  {'estimate':>10}  {'overcount':>10}")
print("-" * 40)
for item in top10:
    true = true_counts[item]
    est  = cms.query(item)
    print(f"{item:>6}  {true:>8}  {est:>10}  {est-true:>10}")

print(f"\nStream length: {cms.stream_length}")
print(f"Max overcount allowed (ε × ||f||₁): {0.005 * cms.stream_length:.0f}")
